# Marketguard ML workspace

Use this notebook to develop the Marketplace fraud detection model(s).
Select **Python (marketguard-ml)** and run all cells in order.

The cells below first import the required libraries, then validate the Conda environment and running Jupyter kernel. A synthetic classification experiment checks that the libraries can work together; it is not a fraud detection model or a measure of real-world performance.


In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import sklearn
from IPython.display import display
from ipykernel.kernelapp import IPKernelApp
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("Successfully imported pandas, NumPy, and scikit-learn.")


Successfully imported pandas, NumPy, and scikit-learn.


In [2]:
expected_environment = "marketguard-ml"
prefix = Path(sys.prefix).resolve()
assert prefix.name == expected_environment, f"Select Python ({expected_environment}); current prefix: {prefix}"
assert (prefix / "conda-meta").is_dir(), f"This is not a Conda environment: {prefix}"
assert IPKernelApp.initialized(), "This cell must execute inside a live Jupyter kernel."
assert get_ipython().kernel is not None, "A live IPython kernel is required."

for module in (pd, np, sklearn):
    assert Path(module.__file__).resolve().is_relative_to(prefix), f"{module.__name__} was imported outside the environment."

print(f"Environment: {prefix.name}")
print(f"Python: {sys.version.split()[0]}")
print(f"Executable: {sys.executable}")
print(f"Kernel: {type(get_ipython()).__name__}")
display(pd.DataFrame({
    "package": ["pandas", "numpy", "scikit-learn"],
    "version": [pd.__version__, np.__version__, sklearn.__version__],
}))
print("PASS: Conda environment, kernel, and package locations are correct.")


Environment: marketguard-ml
Python: 3.12.15
Executable: /Users/franciscomorenoq/miniforge3/envs/marketguard-ml/bin/python
Kernel: ZMQInteractiveShell


,package,version
0,pandas,3.1.0
1,numpy,2.5.4
2,scikit-learn,1.9.1


PASS: Conda environment, kernel, and package locations are correct.


## Library smoke test

Generate reproducible synthetic data, use pandas to prepare the features, and fit a scikit-learn pipeline on a training split. Evaluate on a separate test split and check that prediction probabilities are finite and normalized using NumPy.


In [3]:
X, y = make_classification(
    n_samples=300,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    class_sep=1.5,
    random_state=42,
)
features = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
target = pd.Series(y, name="synthetic_label")
assert not features.isna().any().any()

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42,
)
smoke_test_model = make_pipeline(StandardScaler(), LogisticRegression(random_state=42, max_iter=1000))
smoke_test_model.fit(X_train, y_train)
predictions = smoke_test_model.predict(X_test)
probabilities = smoke_test_model.predict_proba(X_test)

assert predictions.shape == y_test.shape
assert probabilities.shape == (len(y_test), 2)
assert np.isfinite(probabilities).all()
assert ((probabilities >= 0) & (probabilities <= 1)).all()
np.testing.assert_allclose(probabilities.sum(axis=1), 1.0)
assert set(np.unique(predictions)) == {0, 1}

print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")
print(f"Synthetic test accuracy: {accuracy_score(y_test, predictions):.3f}")
print(classification_report(y_test, predictions, zero_division=0))
print("PASS: pandas, NumPy, and scikit-learn work together in this kernel.")


Training rows: 225 | Test rows: 75
Synthetic test accuracy: 0.867
              precision    recall  f1-score   support

           0       0.85      0.89      0.87        37
           1       0.89      0.84      0.86        38

    accuracy                           0.87        75
   macro avg       0.87      0.87      0.87        75
weighted avg       0.87      0.87      0.87        75

PASS: pandas, NumPy, and scikit-learn work together in this kernel.


## Model development

Next, load a labeled listings dataset, define the fraud label and available features, and choose a split that prevents seller or listing leakage. Compare baselines and validate fraud-specific precision and recall before connecting a trained model to the app. No production model is trained or exported by the setup checks above.
